# 🧠 Model Training — GPU Cooling Digital Twin

**Purpose**: Train a LightGBM model for predictive thermal forecasting.

> ⚠️ **SYNTHETIC DATA** — Model trained on synthetic telemetry. Performance metrics apply only to the synthetic domain.

**Targets**:
- `future_temperature_30s` — GPU junction temperature 30 seconds ahead
- `future_power_30s` — GPU power draw 30 seconds ahead

In [ ]:
!pip install lightgbm scikit-learn joblib pandas numpy -q

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
import joblib
import json
import time
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# === LOAD DATASET ===

try:
    from google.colab import drive
    drive.mount('/content/drive')
    df = pd.read_parquet('/content/drive/MyDrive/gpu_cooling_dataset.parquet')
except:
    df = pd.read_parquet('gpu_cooling_dataset.parquet')

print(f'Dataset loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# === DEFINE FEATURES AND TARGETS ===

# Feature columns (exclude targets, timestamp, raw workload string)
exclude_cols = ['timestamp', 'workload', 'future_temperature_30s', 'future_power_30s']
feature_cols = [c for c in df.columns if c not in exclude_cols]

target_cols = ['future_temperature_30s', 'future_power_30s']

X = df[feature_cols].values
y = df[target_cols].values

print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'Targets: {target_cols}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
# === TIME-BASED TRAIN/VAL/TEST SPLIT ===
# Using temporal split to avoid data leakage

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Validation: {X_val.shape[0]:,} samples')
print(f'Test: {X_test.shape[0]:,} samples')

In [ ]:
# === TRAIN LIGHTGBM MODEL ===

params = {
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

model = MultiOutputRegressor(
    lgb.LGBMRegressor(**params)
)

print('Training LightGBM model...')
start_time = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f'Training completed in {train_time:.1f} seconds')

In [ ]:
# === EVALUATE ON TEST SET ===

# Measure inference latency
latencies = []
for _ in range(100):
    sample = X_test[np.random.randint(len(X_test))].reshape(1, -1)
    start = time.perf_counter()
    _ = model.predict(sample)
    latencies.append((time.perf_counter() - start) * 1000)  # ms

avg_latency_ms = np.mean(latencies)

# Predictions on test set
y_pred = model.predict(X_test)

# Temperature metrics
temp_mae = mean_absolute_error(y_test[:, 0], y_pred[:, 0])
temp_rmse = np.sqrt(mean_squared_error(y_test[:, 0], y_pred[:, 0]))
temp_r2 = r2_score(y_test[:, 0], y_pred[:, 0])

# Power metrics
power_mae = mean_absolute_error(y_test[:, 1], y_pred[:, 1])
power_rmse = np.sqrt(mean_squared_error(y_test[:, 1], y_pred[:, 1]))
power_r2 = r2_score(y_test[:, 1], y_pred[:, 1])

print('=== TEST SET METRICS ===')
print(f'\nTemperature Prediction (30s horizon):')
print(f'  MAE:  {temp_mae:.3f} °C')
print(f'  RMSE: {temp_rmse:.3f} °C')
print(f'  R²:   {temp_r2:.4f}')
print(f'\nPower Prediction (30s horizon):')
print(f'  MAE:  {power_mae:.3f} W')
print(f'  RMSE: {power_rmse:.3f} W')
print(f'  R²:   {power_r2:.4f}')
print(f'\nInference Latency: {avg_latency_ms:.2f} ms')

In [ ]:
# === EXPORT MODEL ===

model_path = 'thermal_model.joblib'
metadata_path = 'model_metadata.json'

joblib.dump(model, model_path)

metadata = {
    'model_type': 'LightGBM (MultiOutputRegressor)',
    'target_variables': target_cols,
    'feature_names': feature_cols,
    'prediction_horizon_seconds': 30,
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'metrics': {
        'temperature_mae': round(temp_mae, 4),
        'temperature_rmse': round(temp_rmse, 4),
        'temperature_r2': round(temp_r2, 4),
        'power_mae': round(power_mae, 4),
        'power_rmse': round(power_rmse, 4),
        'power_r2': round(power_r2, 4),
        'inference_latency_ms': round(avg_latency_ms, 4)
    },
    'training_data': 'SYNTHETIC',
    'hyperparameters': params,
    'note': 'Model trained on synthetic GPU telemetry. Metrics apply only to the synthetic domain.'
}

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Model saved to: {model_path}')
print(f'Metadata saved to: {metadata_path}')
print(f'\nDownload these files and place in backend/models/')

In [ ]:
# === OPTIONAL: Save to Google Drive ===
try:
    import shutil
    shutil.copy(model_path, '/content/drive/MyDrive/thermal_model.joblib')
    shutil.copy(metadata_path, '/content/drive/MyDrive/model_metadata.json')
    print('Files saved to Google Drive.')
except:
    print('Google Drive not mounted. Files are in the current directory.')